# 03 — Camada Gold: Star Schema de Kimball

Transforma as tabelas Silver em um modelo dimensional (Kimball) pronto para:
- **Dashboards** e relatórios executivos
- **Análises estatísticas** de desigualdade educacional
- **Treinamento de modelos de ML** (predição de alfabetização)

## Modelo de dados produzido

| Tabela | Tipo | Granularidade | Chave |
|---|---|---|---|
| `gold.tc02_dim_municipio` | Dimensão | 1 linha por município | `sk_municipio` (SK) |
| `gold.tc02_fato_desempenho` | Fato (micro) | 1 linha por aluno/ano | `id_aluno` + `ano` |
| `gold.tc02_fato_alfabetizacao_consolidada` | Fato (macro) | 1 linha por município/rede/ano | `sk_municipio` + `rede` + `ano` |

**Pré-requisito:** executar `02_carga_camada_silver.ipynb`

In [0]:
from pyspark.sql import DataFrame, functions as F, types as T, Window

## Leitura das tabelas Silver

In [0]:
def carregaUltimaParticao(nomeTabela):
    """Lê uma tabela e filtra apenas para o par (ano, mes) de ingestão mais recente."""
    df = spark.read.table(nomeTabela)

    ultima = (
        df.select("_ano_ingestao", "_mes_ingestao")
          .distinct()
          .orderBy(F.col("_ano_ingestao").desc(), F.col("_mes_ingestao").desc())
          .first()
    )

    if ultima is None:
        return df

    return df.filter(
        (F.col("_ano_ingestao") == ultima["_ano_ingestao"]) &
        (F.col("_mes_ingestao") == ultima["_mes_ingestao"])
    )

_tabelas_necessarias = [
    "silver.tc02_dim_municipio",
    "silver.tc02_meta_municipio",
    "silver.tc02_alunos",
]

for _t in _tabelas_necessarias:
    try:
        spark.read.table(_t).limit(1).count()
    except Exception as err:
        raise ValueError(
            f"Tabela Silver '{_t}' não encontrada. Execute 02_carga_camada_silver.ipynb."
        ) from err

df_silver_municipio = carregaUltimaParticao("silver.tc02_dim_municipio")
df_silver_meta_mun  = carregaUltimaParticao("silver.tc02_meta_municipio")
df_silver_alunos    = carregaUltimaParticao("silver.tc02_alunos")

print("Silver carregada com sucesso (última partição).")
print(f"  municipio : {df_silver_municipio.count()} linhas")
print(f"  meta_mun  : {df_silver_meta_mun.count()} linhas")
print(f"  alunos    : {df_silver_alunos.count()} linhas")

## Helper: normalização de `rede`

O Saeb codifica `rede` como inteiro (`1`=Federal, `2`=Estadual, `3`=Municipal, `4`=Privada),
enquanto as tabelas de metas usam rótulos textuais (`MUNICIPAL`, `ESTADUAL` …).
Normalizar para rótulo textual maiúsculo elimina a causa do JOIN vazio na `fato_consolidada`.

In [0]:
def normalizar_rede(df: DataFrame, col_rede: str = "rede") -> DataFrame:
    """
    Converte o código numérico de rede (padrão Saeb/INEP) para o rótulo textual
    usado nas tabelas de metas do Base dos Dados.

    Valores já textuais (ex: 'MUNICIPAL') passam sem alteração via .otherwise().
    """
    return df.withColumn(
        col_rede,
        F.when(F.col(col_rede) == "1", "FEDERAL")
         .when(F.col(col_rede) == "2", "ESTADUAL")
         .when(F.col(col_rede) == "3", "MUNICIPAL")
         .when(F.col(col_rede) == "4", "PRIVADA")
         .otherwise(F.upper(F.trim(F.col(col_rede))))
    )


# Mapa UF: código IBGE (2 dígitos) → sigla e nome completo
_UF_SIGLA: dict = {
    "11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA",
    "16": "AP", "17": "TO", "21": "MA", "22": "PI", "23": "CE",
    "24": "RN", "25": "PB", "26": "PE", "27": "AL", "28": "SE",
    "29": "BA", "31": "MG", "32": "ES", "33": "RJ", "35": "SP",
    "41": "PR", "42": "SC", "43": "RS", "50": "MS", "51": "MT",
    "52": "GO", "53": "DF",
}
_UF_NOME: dict = {
    "RO": "Rondônia",       "AC": "Acre",              "AM": "Amazonas",
    "RR": "Roraima",        "PA": "Pará",              "AP": "Amapá",
    "TO": "Tocantins",      "MA": "Maranhão",          "PI": "Piauí",
    "CE": "Ceará",          "RN": "Rio Grande do Norte","PB": "Paraíba",
    "PE": "Pernambuco",     "AL": "Alagoas",           "SE": "Sergipe",
    "BA": "Bahia",          "MG": "Minas Gerais",       "ES": "Espírito Santo",
    "RJ": "Rio de Janeiro", "SP": "São Paulo",          "PR": "Paraná",
    "SC": "Santa Catarina", "RS": "Rio Grande do Sul",  "MS": "Mato Grosso do Sul",
    "MT": "Mato Grosso",    "GO": "Goiás",             "DF": "Distrito Federal",
}


def _build_uf_expr(mapping: dict, input_col: str) -> F.Column:
    """Constrói uma expressão when/otherwise a partir de um dicionário."""
    expr = F.lit("Desconhecido")
    for k, v in reversed(list(mapping.items())):
        expr = F.when(F.col(input_col) == k, v).otherwise(expr)
    return expr


_expr_sigla_uf = _build_uf_expr(_UF_SIGLA, "id_uf")
_expr_nome_estado = _build_uf_expr(_UF_NOME, "sigla_uf")

## 1. Dimensão Geográfica — `dim_municipio`

Kimball exige uma **Surrogate Key (SK)** estável e inteiramente gerenciada pelo DW,
desacoplada da chave natural IBGE. Usamos `xxhash64` (determinístico) em vez de
`monotonically_increasing_id` (não-determinístico entre execuções) para garantir
idempotência: a mesma cidade sempre recebe a mesma SK, independentemente da ordem
de processamento ou do número de partições.

In [0]:
dim_municipio = (
    df_silver_municipio
    .select("id_municipio")
    .dropDuplicates(["id_municipio"])  
    .withColumn("id_uf",      F.substring(F.col("id_municipio"), 1, 2))
    .withColumn("sigla_uf",   _expr_sigla_uf)
    .withColumn("nome_estado",_expr_nome_estado)
    .withColumn("sk_municipio", F.abs(F.xxhash64(F.col("id_municipio"))).cast(T.LongType()))
    .withColumn("_data_processamento_gold", F.current_timestamp())
    .select(
        "sk_municipio",   
        "id_municipio",  
        "id_uf",
        "sigla_uf",
        "nome_estado",
        "_data_processamento_gold",
    )
)

print(f"dim_municipio: {dim_municipio.count()} municípios únicos")
display(dim_municipio.limit(5))

## 2. Fato Microdados — `fato_desempenho`

Granularidade: **1 linha por aluno por ano**.
Ideal para Feature Engineering em modelos preditivos de alfabetização.
Referencia a SK da `dim_municipio` — não a natural key IBGE.

In [0]:
# Lookup SK: broadcast seguro para dimensão pequena (~5.600 municípios)
_dim_sk_lookup = F.broadcast(
    dim_municipio.select("sk_municipio", "id_municipio")
)

fato_desempenho = (
    normalizar_rede(df_silver_alunos)
    .dropna(subset=["id_aluno", "ano"])
    .join(_dim_sk_lookup, on="id_municipio", how="left")
    .select(
        F.col("id_aluno"),
        F.col("ano").cast(T.IntegerType()).alias("ano"),
        F.col("sk_municipio"),
        F.col("id_escola"),
        F.col("caderno"),
        F.col("serie"),
        F.col("rede"),
        F.col("presenca"),
        F.col("preenchimento_caderno"),
        F.col("alfabetizado").cast(T.IntegerType()).alias("alfabetizado"),
        F.round(F.col("proficiencia").cast(T.DoubleType()), 4).alias("proficiencia"),
        F.round(F.col("peso_prova_portugues").cast(T.DoubleType()),   4).alias("peso_prova_portugues"),
        F.current_timestamp().alias("_data_processamento_gold"),
    )
)

print(f"fato_desempenho: {fato_desempenho.count()} registros")
display(fato_desempenho.limit(5))

## 3. Fato Consolidado — `fato_alfabetizacao_consolidada`

Granularidade: **1 linha por município × rede × ano**.

### Por que a versão anterior gerava tabela vazia — dois bugs simultâneos:

**Bug 1 (causa raiz):** A tabela `meta_municipio` tem `ano = 2023` (ano-base da Pesquisa
Alfabetiza Brasil). O INNER JOIN em `["ano", "id_municipio", "rede"]` cruzava `2023 ≠ 2024`
(ano dos alunos) → 0 linhas. A coluna de metas **não participa do JOIN temporal**; o `ano`
correto para selecionar a coluna-meta (`meta_alfabetizacao_2024` etc.) é o `ano` dos alunos,
avaliado com `when()` **após** o JOIN.

**Bug 2 (agravante):** `rede` codificada como `"3"` nos alunos (código Saeb) vs `"MUNICIPAL"`
nas metas (rótulo textual). O JOIN em `rede` também retornaria 0 linhas independentemente
do Bug 1.

In [0]:
# Normaliza rede antes da agregação para que o valor resultante ('MUNICIPAL' etc.)
# bata com a encoding da tabela de metas no JOIN seguinte.
df_alunos_agg = (
    normalizar_rede(df_silver_alunos)
    .filter(F.col("presenca") == "1")          # considera apenas alunos presentes
    .groupBy("ano", "id_municipio", "rede")
    .agg(
        F.count("id_aluno").alias("total_alunos_avaliados"),
        F.avg(
            F.col("proficiencia").cast(T.DoubleType())
        ).alias("media_proficiencia_alunos"),
        # avg(flag_binária) = proporção [0.0, 1.0]; ×10 mapeia para escala [0, 10]
        (
            F.avg(F.col("alfabetizado").cast(T.DoubleType())) * 10.0
        ).alias("taxa_alfabetizacao_real_escala_10"),
    )
)

# A meta_municipio tem 'ano = 2023' (ano-base da pesquisa) e colunas
# meta_alfabetizacao_2024..2030 com as metas futuras.
# Usamos window para garantir 1 linha por (id_municipio, rede), mantendo
# a projeção mais recente se houver múltiplas safras de dados.
_w_latest_meta = Window.partitionBy("id_municipio", "rede").orderBy(F.desc("ano"))

df_meta_lookup = (
    df_silver_meta_mun
    .withColumn("_rank", F.row_number().over(_w_latest_meta))
    .filter(F.col("_rank") == 1)
    .drop("_rank", "ano")                
    .select(
        "id_municipio",
        "rede",
        "meta_alfabetizacao_2024",
        "meta_alfabetizacao_2025",
        "meta_alfabetizacao_2026",
        "meta_alfabetizacao_2027",
        "meta_alfabetizacao_2028",
        "meta_alfabetizacao_2029",
        "meta_alfabetizacao_2030",
        "percentual_participacao",
        "nivel_alfabetizacao",
    )
)

# O 'ano' dos alunos (ex: 2024) é usado para selecionar a coluna-meta correta
fato_alfabetizacao_consolidada = (
    df_alunos_agg
    .join(
        F.broadcast(df_meta_lookup),            # broadcast: metas são << que microdados
        on=["id_municipio", "rede"],
        how="inner",
    )
    .withColumn(
        "meta_alfabetizacao_projetada",
        F.when(F.col("ano") == 2024, F.col("meta_alfabetizacao_2024"))
         .when(F.col("ano") == 2025, F.col("meta_alfabetizacao_2025"))
         .when(F.col("ano") == 2026, F.col("meta_alfabetizacao_2026"))
         .when(F.col("ano") == 2027, F.col("meta_alfabetizacao_2027"))
         .when(F.col("ano") == 2028, F.col("meta_alfabetizacao_2028"))
         .when(F.col("ano") == 2029, F.col("meta_alfabetizacao_2029"))
         .when(F.col("ano") == 2030, F.col("meta_alfabetizacao_2030"))
         .otherwise(F.lit(None).cast(T.DoubleType()))  # anos fora do programa: sem meta
    )
    .withColumn(
        "desvio_da_meta",
        F.col("taxa_alfabetizacao_real_escala_10") - F.col("meta_alfabetizacao_projetada"),
    )
    .join(_dim_sk_lookup, on="id_municipio", how="left")
    .dropna(subset=["total_alunos_avaliados"])
    .select(
        F.col("sk_municipio"),
        F.col("ano").cast(T.IntegerType()).alias("ano"),
        F.col("rede"),
        F.col("total_alunos_avaliados"),
        F.round(F.col("media_proficiencia_alunos"),         2).alias("media_proficiencia_alunos"),
        F.round(F.col("taxa_alfabetizacao_real_escala_10"),  2).alias("taxa_alfabetizacao_real_escala_10"),
        F.round(F.col("meta_alfabetizacao_projetada"),       2).alias("meta_alfabetizacao_projetada"),
        F.round(F.col("desvio_da_meta"),                     2).alias("desvio_da_meta"),
        F.round(F.col("percentual_participacao"),            2).alias("meta_percentual_participacao"),
        F.col("nivel_alfabetizacao"),
        F.current_timestamp().alias("_data_processamento_gold"),
    )
)

print(f"fato_alfabetizacao_consolidada: {fato_alfabetizacao_consolidada.count()} registros")
display(fato_alfabetizacao_consolidada.limit(5))

## 4. Escrita na Camada Gold

- `dim_municipio`: sem partição (dimensão estática, leitura integral em queries de BI)
- Tabelas fato: particionadas por `ano` para habilitar Partition Pruning (FinOps)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

# (nome_tabela, DataFrame, particionar_por_ano)
_tabelas_gold: list = [
    ("gold.tc02_dim_municipio",                  dim_municipio,                   False),
    ("gold.tc02_fato_desempenho",                fato_desempenho,                 True),
    ("gold.tc02_fato_alfabetizacao_consolidada", fato_alfabetizacao_consolidada,  True),
]

for nome, df, partizionar in _tabelas_gold:
    writer = (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
    )
    if partizionar:
        writer = writer.partitionBy("ano")
    writer.saveAsTable(nome)
    print(f"  ✓ {nome}: {spark.read.table(nome).count()} linhas")

print("\nCamada Gold gravada com sucesso.")

## 5. Relatório de Qualidade — Gold

Validações mínimas que devem passar para o pipeline ser considerado saudável:
- Nenhuma tabela Gold pode estar vazia
- Nenhum fato pode ter `sk_municipio = NULL` (quebraria toda query de BI)
- `taxa_alfabetizacao_real_escala_10` deve estar no intervalo [0, 10]

In [0]:
def _check(label: str, passed: bool, detail: str = "") -> None:
    status = "OK  " if passed else "FALHA"
    msg = f"[{status}] {label}"
    if detail:
        msg += f" — {detail}"
    print(msg)
    if not passed:
        raise ValueError(msg)

print("=" * 60)
print("RELATÓRIO DE QUALIDADE — CAMADA GOLD")
print("=" * 60)

for _nome in [n for n, _, _ in _tabelas_gold]:
    _cnt = spark.read.table(_nome).count()
    _check(f"{_nome} não está vazia", _cnt > 0, f"{_cnt} linhas")

_fato_c = spark.read.table("gold.tc02_fato_alfabetizacao_consolidada")
_sem_sk = _fato_c.filter(F.col("sk_municipio").isNull()).count()
_check(
    "fato_consolidada sem sk_municipio",
    _sem_sk == 0,
    f"{_sem_sk} linhas órfãs",
)
_fora_escala = _fato_c.filter(
    (F.col("taxa_alfabetizacao_real_escala_10") < 0) |
    (F.col("taxa_alfabetizacao_real_escala_10") > 10)
).count()
_check(
    "taxa_alfabetizacao dentro de [0, 10]",
    _fora_escala == 0,
    f"{_fora_escala} linhas fora da escala",
)

print("=" * 60)
print("Pipeline Gold concluído com sucesso.")
print("Próximo passo: executar 04_streaming_ingestion.py")